In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "perfumes.csv"
)

In [4]:
df_model = df[
    (df["vote_count"] >= 50) &
    (df["rating_avg"] > 0)
].copy()

print(df_model.shape)

(36732, 59)


In [5]:
df_model[
    [
        "rating_avg",
        "longevity_avg",
        "sillage_avg",
        "price_value_avg",
        "have",
        "want",
        "year"
    ]
].describe()

,rating_avg,longevity_avg,sillage_avg,price_value_avg,have,want,year
count,36732.000000,36732.000000,36732.000000,36732.000000,36732.000000,36732.000000,34035.000000
mean,3.968042,3.188279,2.281799,3.289486,761.404634,623.828869,2014.048421
std,0.277339,0.496035,0.360767,0.728840,2355.560194,1531.592267,13.839156
min,1.407500,1.000000,1.125700,0.000000,3.000000,1.000000,1533.000000
25%,3.800000,2.850675,2.032300,2.758900,101.000000,113.000000,2011.000000
50%,3.981800,3.187200,2.275900,3.250000,203.000000,227.000000,2017.000000
75%,4.150950,3.538500,2.527725,3.883075,528.000000,528.000000,2022.000000
max,4.925900,4.896100,3.902300,5.000000,59557.000000,42971.000000,2026.000000


In [6]:
numeric_cols = [
    "rating_avg",
    "longevity_avg",
    "sillage_avg",
    "price_value_avg",
    "have",
    "want",
    "year"
]

corr = df_model[numeric_cols].corr()

corr["rating_avg"].sort_values(
    ascending=False
)

rating_avg         1.000000
longevity_avg      0.284891
sillage_avg        0.211380
want               0.131972
have               0.088505
price_value_avg    0.036205
year              -0.082843
Name: rating_avg, dtype: float64

Longevity is the strongest driver with corr = 0.285, the community seems to reward perfumes that last longer. Sillage matters too, stronger projection tends to be associated with higher ratings. Surprisingly, value-for-money barely matters. Another intresting discovery is that newer perfumes slightly underperform. Older classics may have stronger reputations.

In [8]:
top_accords = [
    "woody",
    "sweet",
    "powdery",
    "citrus",
    "aromatic",
    "fresh spicy",
    "floral",
    "fruity",
    "warm spicy",
    "amber",
    "musky",
    "white floral",
    "vanilla",
    "fresh",
    "green",
    "rose",
    "earthy",
    "balsamic",
    "animalic",
    "patchouli"
]
for accord in top_accords:

    col_name = accord.replace(" ", "_")

    df_model[col_name] = (
        df_model["accords"]
        .fillna("")
        .str.contains(
            accord,
            case=False
        )
        .astype(int)
    )

In [9]:
df_model[
    [
        "amber",
        "vanilla",
        "citrus",
        "woody"
    ]
].head()

,amber,vanilla,citrus,woody
0,0,0,1,0
1,0,0,0,1
2,0,1,0,1
3,0,0,1,0
4,1,0,0,1


In [10]:
accord_effects = []

for accord in top_accords:

    col = accord.replace(" ", "_")

    avg_rating = (
        df_model
        .groupby(col)["rating_avg"]
        .mean()
    )

    accord_effects.append(
        {
            "accord": accord,
            "rating_without": avg_rating[0],
            "rating_with": avg_rating[1],
            "difference":
                avg_rating[1] - avg_rating[0]
        }
    )

accord_effects = pd.DataFrame(
    accord_effects
)

accord_effects.sort_values(
    "difference",
    ascending=False
)

,accord,rating_without,rating_with,difference
9,amber,3.944053,4.014479,0.070427
17,balsamic,3.959323,4.022681,0.063358
8,warm spicy,3.947806,4.004504,0.056698
12,vanilla,3.953531,4.003065,0.049534
16,earthy,3.963910,3.992364,0.028453
4,aromatic,3.956781,3.983143,0.026362
5,fresh spicy,3.958679,3.984164,0.025485
0,woody,3.952634,3.975912,0.023278
18,animalic,3.965389,3.987799,0.022410
2,powdery,3.966822,3.969276,0.002454


Perfumes containing amber, balsamic, warm spicy, and vanilla accords consistently receive higher community ratings. Conversely, fruity and floral accords are associated with lower average ratings among Fragrantica users.

In [13]:
from scipy.stats import ttest_ind

amber_yes = df_model[
    df_model["fruity"] == 1
]["rating_avg"]

amber_no = df_model[
    df_model["fruity"] == 0
]["rating_avg"]

t_stat, p_value = ttest_ind(
    amber_yes,
    amber_no,
    equal_var=False
)

print("p-value:", p_value)

p-value: 1.1309855517976552e-98


The lower ratings associated with fruity accords are also statistically significant and not due to random chance.

In [14]:
from scipy.stats import ttest_ind

amber_yes = df_model[
    df_model["amber"] == 1
]["rating_avg"]

amber_no = df_model[
    df_model["amber"] == 0
]["rating_avg"]

t_stat, p_value = ttest_ind(
    amber_yes,
    amber_no,
    equal_var=False
)

print("p-value:", p_value)

p-value: 7.70682073214358e-119


There is overwhelming statistical evidence that perfumes containing the amber accord have different average ratings from perfumes without amber.

In [15]:
from scipy.stats import ttest_ind
import pandas as pd

test_accords = [
    "amber",
    "balsamic",
    "warm_spicy",
    "vanilla",
    "earthy",
    "fruity",
    "floral",
    "musky",
    "green",
    "rose"
]

ttest_results = []

for accord in test_accords:
    yes = df_model[df_model[accord] == 1]["rating_avg"]
    no = df_model[df_model[accord] == 0]["rating_avg"]

    t_stat, p_value = ttest_ind(
        yes,
        no,
        equal_var=False
    )

    ttest_results.append({
        "accord": accord,
        "mean_with_accord": yes.mean(),
        "mean_without_accord": no.mean(),
        "rating_difference": yes.mean() - no.mean(),
        "p_value": p_value
    })

ttest_results = pd.DataFrame(ttest_results)

ttest_results.sort_values(
    "rating_difference",
    ascending=False
)

,accord,mean_with_accord,mean_without_accord,rating_difference,p_value
0,amber,4.014479,3.944053,0.070427,7.706821e-119
1,balsamic,4.022681,3.959323,0.063358,1.466002e-53
2,warm_spicy,4.004504,3.947806,0.056698,8.466852e-78
3,vanilla,4.003065,3.953531,0.049534,1.127508e-55
4,earthy,3.992364,3.963910,0.028453,2.728165e-12
9,rose,3.943620,3.974639,-0.031019,4.276024e-19
8,green,3.944121,3.975413,-0.031292,2.164967e-20
7,musky,3.942762,3.979666,-0.036904,1.490847e-31
6,floral,3.944494,3.994556,-0.050061,1.441469e-66
5,fruity,3.928063,3.990984,-0.062921,1.130986e-98


In [16]:
ttest_results.to_csv(
    "accord_rating_ttest_results.csv",
    index=False
)

### Random Forest Model

In [19]:
X = df_model[features].copy()
y = df_model["rating_avg"].copy()

X["year"] = X["year"].fillna(X["year"].median())

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(0)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 4))
print("R²:", round(r2_score(y_test, pred), 4))

MAE: 0.1932
R²: 0.1746


In [21]:
importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

importance.head(20)

,feature,importance
0,longevity_avg,0.243724
2,price_value_avg,0.184380
1,sillage_avg,0.169179
3,year,0.121382
7,citrus,0.016568
14,musky,0.016392
6,powdery,0.016021
4,woody,0.015538
12,warm_spicy,0.015362
5,sweet,0.015091


In [18]:
X.isna().sum().sort_values(ascending=False)

year               2697
longevity_avg         0
amber                 0
animalic              0
balsamic              0
earthy                0
rose                  0
green                 0
fresh                 0
vanilla               0
white_floral          0
musky                 0
warm_spicy            0
sillage_avg           0
fruity                0
floral                0
fresh_spicy           0
aromatic              0
citrus                0
powdery               0
sweet                 0
woody                 0
price_value_avg       0
patchouli             0
dtype: int64